# RRAM LTD: constant-hold pulses and independent low-bias reads

Run this notebook from the repository root. The full default sweep is 7 × 500 pulses;

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
from pathlib import Path
from datetime import datetime
import json
from experiments.ltp_ltd.run import settings, initial_state, run_condition, ROOT
from experiments.ltp_ltd.common import json_value
from analysis.ltp_ltd.plot import plot_curves
from IPython.display import display, Image


## 1. Pulse and output parameters

Each voltage starts independently from the same packaged initial state. 

In [ ]:
DIRECTION = "LTD"
VOLTAGES = [-0.5, -0.7, -0.8, -0.9, -0.95, -1.0, -1.1]
PULSES = 500
HOLD_TIME = 1e-6
READ_TIME = 1e-10
CHECKPOINT_INTERVAL = 50
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT = ROOT / "results_ltp_ltd" / RUN_ID / DIRECTION


## 2. Inspect actual saved configuration and material parameters

The loaded simulator owns the active `conf`; creating another dictionary does not override it. The exact reference configuration is printed below. 


In [ ]:
preview = initial_state(DIRECTION)
try:
    print(json.dumps(preview.conf, default=json_value, indent=2))
    print(json.dumps({r.name: vars(r.material) for r in preview.mesh.regions}, default=json_value, indent=2))
finally:
    preview.close()


## 3. Program, independently read, and export each pulse

`run_condition` uses `voltage_hold_solver` with one initial step, retry halving and full state rollback. Each accepted programming pulse exports VTK/PVD. 

In [ ]:
condition_paths = []
for voltage in VOLTAGES:
    simulator = initial_state(DIRECTION)
    path = run_condition(simulator, DIRECTION, voltage, OUTPUT / f"{voltage:g}V",
                         pulses=PULSES, hold_time=HOLD_TIME, read_time=READ_TIME,
                         checkpoint_interval=CHECKPOINT_INTERVAL, plot_every=10)
    condition_paths.append(path)


## 4. Plot the calculated independent-read conductance

Pure lines with the reference voltage colors. PNG/TIFF/PDF and plotting CSV/metadata are saved. 

In [ ]:
figure = plot_curves(condition_paths, OUTPUT / "figures", DIRECTION)
display(Image(filename=str(figure)))
print("Results:", OUTPUT)
